# 🧠 Ferni Tool Router - LFM2.5 (Liquid AI)

Train a state-of-the-art tool classifier using Liquid AI's LFM2.5-1.2B model.

**Requirements:**
- Runtime: T4 GPU
- Time: ~40-60 minutes
- Output: `ferni-router-lfm.zip`

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers datasets accelerate peft bitsandbytes scikit-learn tqdm

In [ ]:
# Cell 2: Upload training files
from google.colab import files
import os

print("📤 Please upload these files:")
print("  1. train.jsonl")
print("  2. validation.jsonl")
print("  3. label_map.json")
print("")

uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files")

In [ ]:
# Cell 3: Check GPU
import torch
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️ GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4: Load data
import json
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

print("📂 Loading training data...")

with open('train.jsonl', 'r') as f:
    train_data = [json.loads(line) for line in f]

with open('validation.jsonl', 'r') as f:
    val_data = [json.loads(line) for line in f]

with open('label_map.json', 'r') as f:
    label_map = json.load(f)

print(f"✅ Training examples: {len(train_data)}")
print(f"✅ Validation examples: {len(val_data)}")
print(f"✅ Number of labels: {len(label_map)}")

# Prepare labels
all_tools = list(label_map.keys())
mlb = MultiLabelBinarizer(classes=all_tools)
mlb.fit([all_tools])

def get_labels(examples):
    labels = []
    for ex in examples:
        tools = ex.get('selected_tools', ex.get('tools', []))
        if isinstance(tools, str):
            tools = [tools]
        labels.append(tools)
    return mlb.transform(labels)

train_labels = get_labels(train_data)
val_labels = get_labels(val_data)
print(f"✅ Labels prepared")

In [ ]:
# Cell 5: Load LFM2.5 model with custom classifier head
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print("🔧 Loading LFM2.5-1.2B-Base model...")

model_name = "LiquidAI/LFM2.5-1.2B-Base"
num_labels = len(label_map)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Custom classifier model
class LFM2Classifier(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.base = AutoModel.from_pretrained(
            base_model_name,
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
        hidden_size = self.base.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        # Pool last token (like GPT-style)
        hidden_states = outputs.last_hidden_state
        batch_size = input_ids.shape[0]
        # Get the last non-padded token position
        sequence_lengths = attention_mask.sum(dim=1) - 1
        pooled = hidden_states[torch.arange(batch_size, device=hidden_states.device), sequence_lengths]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled.float())  # Cast to float32 for classifier

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())

        return {'loss': loss, 'logits': logits}

model = LFM2Classifier(model_name, num_labels)
model = model.cuda()

# Apply LoRA for efficient fine-tuning
print("🔧 Applying LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)
model.base = get_peft_model(model.base, lora_config)
model.base.print_trainable_parameters()

print(f"✅ Model loaded!")

In [ ]:
# Cell 6: Prepare datasets
from torch.utils.data import Dataset, DataLoader

class ToolDataset(Dataset):
    def __init__(self, examples, labels, tokenizer, max_length=128):
        self.queries = [ex['query'] for ex in examples]
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.queries[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': self.labels[idx]
        }

train_dataset = ToolDataset(train_data, train_labels, tokenizer)
val_dataset = ToolDataset(val_data, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Val batches: {len(val_loader)}")

In [ ]:
# Cell 7: Training loop
from tqdm.auto import tqdm
import torch.optim as optim

print("🏋️ Starting training...")

optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 1
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].cuda()
        attention_mask = batch['attention_mask'].cuda()
        labels = batch['labels'].cuda()

        outputs = model(input_ids, attention_mask, labels)
        loss = outputs['loss']

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        progress.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            input_ids = batch['input_ids'].cuda()
            attention_mask = batch['attention_mask'].cuda()
            labels = batch['labels'].cuda()

            outputs = model(input_ids, attention_mask, labels)
            val_loss += outputs['loss'].item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"\n📊 Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print("💾 Saving best model...")
        torch.save(model.state_dict(), 'best_model.pt')

print("\n✅ Training complete!")

In [ ]:
# Cell 8: Save and download model
import shutil
import os

print("💾 Saving model...")

# Create output directory
os.makedirs('ferni-router-lfm', exist_ok=True)

# Save model weights
model.load_state_dict(torch.load('best_model.pt'))
torch.save(model.state_dict(), 'ferni-router-lfm/model.pt')

# Save tokenizer
tokenizer.save_pretrained('ferni-router-lfm')

# Save label map
with open('ferni-router-lfm/label_map.json', 'w') as f:
    json.dump(label_map, f)

# Save config
config = {
    'base_model': 'LiquidAI/LFM2.5-1.2B-Base',
    'num_labels': num_labels,
    'max_length': 128,
}
with open('ferni-router-lfm/config.json', 'w') as f:
    json.dump(config, f)

# Create zip
shutil.make_archive('ferni-router-lfm', 'zip', 'ferni-router-lfm')

print("\n📥 Downloading model...")
files.download('ferni-router-lfm.zip')

print("\n✅ Done! Extract to models/ferni-router-lfm/")